# Data Migration: SQL Server to Postgres

In [131]:
import os
import pandas as pd
import pyodbc
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv

In [132]:
os.getcwd()

'c:\\Users\\opeab\\OneDrive\\Documents\\Github\\sql-server-to-postgres-migration'

In [133]:
os.listdir()

['.env',
 '.git',
 '.gitignore',
 '.venv',
 'generate_data.py',
 'README.md',
 'run_migration_uat.ipynb']

In [134]:
os.path.isfile(".env")

True

## 1. Load credentials

In [135]:
load_dotenv(".env")

True

In [136]:
sql_host = os.getenv("SQL_SERVER_HOST")
sql_db = os.getenv("SQL_SERVER_DB")

In [137]:
print(f"SQL SERVER HOST: {sql_host}")
print(f"SQL SERVER DB: {sql_db}")

SQL SERVER HOST: OPSY\SQLEXPRESS
SQL SERVER DB: TransactionDB_UAT


In [138]:
pg_host = os.getenv("POSTGRES_HOST")
pg_port = os.getenv("POSTGRES_PORT")
pg_db = os.getenv("POSTGRES_DB")
pg_user = os.getenv("POSTGRES_USER")
pg_password = os.getenv("POSTGRES_PASSWORD")

In [139]:
print(f"POSTGRES HOST: {pg_host}")
print(f"POSTGRES PORT: {pg_port}")
print(f"POSTGRES DB: {pg_db}")
print(f"POSTGRES USER: {pg_user}")
print(f"POSTGRES PASSWORD: {pg_password}")

POSTGRES HOST: localhost
POSTGRES PORT: 5432
POSTGRES DB: transaction_uat
POSTGRES USER: postgres
POSTGRES PASSWORD: Positive*1


## 2. Connect to SQL Server

In [140]:
print("Connecting  to SQL Server...")
print(f"   Server: {sql_host}")
print(f"   Database: {sql_db}")

Connecting  to SQL Server...
   Server: OPSY\SQLEXPRESS
   Database: TransactionDB_UAT


In [141]:
try:
    sql_conn_string = (
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={sql_host};"
        f"DATABASE={sql_db};"
        "Trusted_Connection=yes;"
    )

    sql_conn = pyodbc.connect(sql_conn_string)
    sql_cursor = sql_conn.cursor()
    print("[SUCCESS] SQL Server connection established.")

except Exception as e:
    print(f"SQL Server connection failed: {e}")
    print(""" How to troubleshoot:
          > 1. Check server name in .env file is correct
          . 2. Verify SQL Server is running
          > 3. Check Windows Authentication is enabled
            ....
""")

[SUCCESS] SQL Server connection established.


# 3. Connect to PostgreSQL

In [142]:
print("Connecting to PostgreSQL...")
print(f"    Server: {pg_host}")
print(f"    Database: {pg_db}")

Connecting to PostgreSQL...
    Server: localhost
    Database: transaction_uat


In [143]:
try: 
    pg_conn = psycopg2.connect(
        host=pg_host,
        port=pg_port,
        database=pg_db,
        user=pg_user,
        password=pg_password
    )

    pg_cursor=pg_conn.cursor()
    pg_cursor.execute("SELECT version();")

    pg_version = pg_cursor.fetchone()[0]

    print("Connected to PostgreSQL successfully!")
    print(f"    Version: {pg_version[:50]}...\n")


except psycopg2.OperationalError as e:
    print(f"Postgres connection failed:{e}")
    print(""" How to troubleshoot:
          > 1. Check Postgres is running
          > 2. Verify username + password
          > 3. Check database exists
        ....

""")
    
except Exception as e:
    print(f" Unexpected error: {e}")
    raise

Connected to PostgreSQL successfully!
    Version: PostgreSQL 18.1 on x86_64-windows, compiled by msv...



# 4. Define the tables to migrate

### Migration order

- Categories (no dependencies)
- Supplies (no dependencies)
- Customers (no dependencies)
- Products (depends on Categories and Suppliers)


In [144]:
tables_to_migrate = ['Categories', 'Suppliers', 'Customers', 'Products']
print(tables_to_migrate)

['Categories', 'Suppliers', 'Customers', 'Products']


In [145]:
print("Table to migrate:")
for i, table in enumerate(tables_to_migrate, 1):
    print(f"    {i}. {table}")

total_no_tbls = len(tables_to_migrate)
print(f"\nTotal no of tables to migrate: {total_no_tbls}")

Table to migrate:
    1. Categories
    2. Suppliers
    3. Customers
    4. Products

Total no of tables to migrate: 4


# 5. Run pre-migration checks

In [146]:
print("=" * 50)
print(">>> Check 1: ROW COUNTS")
print("=" * 50)

>>> Check 1: ROW COUNTS


In [147]:
baseline_counts = {}


try:
    for table in tables_to_migrate:
        quoted_table = f"[{table}]"
        row_count_query = f"SELECT COUNT(*) as total_rows FROM {quoted_table};"
        sql_cursor.execute(row_count_query)
        count = sql_cursor.fetchone()[0]

        baseline_counts[table] = count
        print(f"{table:15} {count:>12} rows")

    total_rows = sum(baseline_counts.values())
    print(f"{'-' * 30}")
    print(f"{'TOTAL':15} {total_rows:>12} rows")
    print("\n Baseline captured! ")

except Exception as e:
    print(f"Failed to get baseline counts: {e}")
    raise

Categories                 8 rows
Suppliers               5000 rows
Customers             900000 rows
Products              150000 rows
------------------------------
TOTAL                1055008 rows

 Baseline captured! 


In [148]:
tables_to_migrate = {'Categories', 'Suppliers', 'Customers', 'Products'}
print(tables_to_migrate)

{'Products', 'Suppliers', 'Customers', 'Categories'}


In [149]:
print("Table to migrate:")
for i, table in enumerate(tables_to_migrate, 1):
    print(f"    {i}.  {table}")

print(f"\nTotal no of tables to migrate: {len(tables_to_migrate)}")

Table to migrate:
    1.  Products
    2.  Suppliers
    3.  Customers
    4.  Categories

Total no of tables to migrate: 4


# 5. Run pre-migration checks

In [150]:
print("=" * 50)
print(">>> ROW COUNTS")
print("=" * 50)

>>> ROW COUNTS


In [151]:
test_query = "SELECT COUNT(*) AS total_rows FROM Categories;"
sql_cursor.execute(test_query)

count = sql_cursor.fetchone()[0]

print(f"Results: {count}")

Results: 8


In [152]:
baseline_counts ={}


try:
    for table in tables_to_migrate:
        row_count_query = f"SELECT COUNT(*) AS total_rows FROM {table}"#(Warning: Do not input SQL queries with f-strings in production. Malicious input can lead to SQL injection attacks. Always use parameterized queries or proper sanitization.)
        sql_cursor.execute(row_count_query)
        count = sql_cursor.fetchone()[0]

        baseline_counts[table] = count
        print(f"{table:15} {count:>12} rows")
        
        
    baseline_counts[table] = count
    print(f"{'-' * 30}")
    print(f"{'TOTAL':15} {total_rows:>12} rows")
    print("\n Baseline captured! ")

except Exception as e:
    print("Failed to get baseline counts: {e}")
    raise

Products              150000 rows
Suppliers               5000 rows
Customers             900000 rows
Categories                 8 rows
------------------------------
TOTAL                1055008 rows

 Baseline captured! 


In [153]:
print("=" * 50)
print(">>> Check 2: NULL COUNTS (CustomerName)")
print("=" * 50)

quality_issues = []

>>> Check 2: NULL COUNTS (CustomerName)


In [154]:
try:
    print("\nCheck 2: NULL COUNTS (CustomerName)")
    sql_cursor.execute("""SELECT COUNT(*) AS null_count
                            FROM Customers
                            WHERE CustomerName IS NULL""")
    null_names = sql_cursor.fetchone()[0]
    if null_names > 0:
        quality_issues.append(f"    > {null_names:,} customers with NULL names...")
    # print(quality_issues)

    print("\nCHECK 3: Invalid email format check")
    sql_cursor.execute("""SELECT COUNT(*) AS invalid_email_count
                            FROM Customers
                            WHERE Email LIKE '%@invalid'   """)
    invalid_emails = sql_cursor.fetchone()[0]
    if invalid_emails > 0:
        quality_issues.append(f"    > {invalid_emails:,} email with invalid email formats...")
    # print(quality_issues)

    print("\nCHECK 4: NEGATIVE PRODUCT PRICES")
    sql_cursor.execute("""SELECT COUNT(*) AS negative_price_count
                            FROM Products
                            WHERE UnitPrice < 0""")
    negative_price = sql_cursor.fetchone()[0]
    if negative_price > 0:
        quality_issues.append(f"    > {negative_price:,} prices contain negative values...")
    # print(quality_issues)

    print("\nCHECK 5: NEGATIVE STOCK QUANTITIES")
    sql_cursor.execute("""SELECT COUNT(*) AS negative_stock_quantities_count
                            FROM Products
                            WHERE StockQuantity < 0
                        """ )
    negative_stock_quantities = sql_cursor.fetchone()[0]
    if negative_stock_quantities > 0:
        quality_issues.append(f"    > {negative_stock_quantities:,} products with negative stock...")
    #print(quality_issues)

    print("\nCHECK 6: ORPHANED FOREIGN KEYS")
    sql_cursor.execute("""SELECT COUNT(*) AS orphaned_records
                            FROM Products prod
                            WHERE NOT EXISTS (SELECT 1
                                                FROM Suppliers sup
                                                WHERE sup.SupplierID = prod.SupplierID)
                       """)
    orphaned_fks = sql_cursor.fetchone()[0]
    if orphaned_fks > 0:
        quality_issues.append(f"    > {orphaned_fks:,} products with orphaned foreign keys...")
    # print(quality_issues)


    print("\nCHECK 7: FUTURE DATES CHECK")
    sql_cursor.execute("""SELECT COUNT(*) AS future_dates_count
                            FROM Customers
                            WHERE CreatedDate > GETDATE()
                       """)
    future_dates = sql_cursor.fetchone()[0]
    if future_dates > 0:
        quality_issues.append(f"    > {future_dates:,} customer with future creations data later than current date...")
    # print(quality_issues)

    if quality_issues:
        print("\nData quality issues found (will migrate as-is)")
        for issue in quality_issues:
            print(issue)
    else: 
        print("No data quality issues identified!")


except Exception as e:
    print(f"[ERROR] ===> Unexpected issue: {e}")
    raise


Check 2: NULL COUNTS (CustomerName)

CHECK 3: Invalid email format check

CHECK 4: NEGATIVE PRODUCT PRICES

CHECK 5: NEGATIVE STOCK QUANTITIES

CHECK 6: ORPHANED FOREIGN KEYS

CHECK 7: FUTURE DATES CHECK

Data quality issues found (will migrate as-is)
    > 4,514 customers with NULL names...
    > 8,844 email with invalid email formats...
    > 775 prices contain negative values...
    > 1,467 products with negative stock...
    > 24,700 products with orphaned foreign keys...
    > 7,050 customer with future creations data later than current date...


# 6. Get table schema

In [155]:
print("=" * 65)
print("ANALYZE TABLE SCHEMA")
print("=" * 65)

ANALYZE TABLE SCHEMA


In [156]:
table_schema = {}


try:
    for table in tables_to_migrate:
        schema_query = f"""
            SELECT
                COLUMN_NAME,
                DATA_TYPE,
                CHARACTER_MAXIMUM_LENGTH,
                IS_NULLABLE
            FROM 
                INFORMATION_SCHEMA.COLUMNS
            WHERE
                table_name = '{table}'
            ORDER BY
                ORDINAL_POSITION
"""
        schema_df = pd.read_sql(schema_query, sql_conn)
        print(f"\n{table}")
        print("-" * 10)
        print(schema_df)
        table_schema[table] = schema_df
        print("\n\n\n")

    

except Exception as e:
    pass

C:\Users\opeab\AppData\Local\Temp\ipykernel_22836\2925528738.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  schema_df = pd.read_sql(schema_query, sql_conn)



Products
----------
     COLUMN_NAME DATA_TYPE  CHARACTER_MAXIMUM_LENGTH IS_NULLABLE
0      ProductID       int                       NaN          NO
1    ProductName  nvarchar                     200.0         YES
2     CategoryID       int                       NaN         YES
3     SupplierID       int                       NaN         YES
4      UnitPrice     money                       NaN         YES
5  StockQuantity       int                       NaN         YES
6    CreatedDate  datetime                       NaN         YES





Suppliers
----------
    COLUMN_NAME DATA_TYPE  CHARACTER_MAXIMUM_LENGTH IS_NULLABLE
0    SupplierID       int                       NaN          NO
1  SupplierName  nvarchar                     150.0         YES
2   ContactName  nvarchar                     100.0         YES
3       Country  nvarchar                     100.0         YES
4         Phone  nvarchar                      20.0         YES





Customers
----------
    COLUMN_NAME DATA_TY

C:\Users\opeab\AppData\Local\Temp\ipykernel_22836\2925528738.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  schema_df = pd.read_sql(schema_query, sql_conn)
C:\Users\opeab\AppData\Local\Temp\ipykernel_22836\2925528738.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  schema_df = pd.read_sql(schema_query, sql_conn)
C:\Users\opeab\AppData\Local\Temp\ipykernel_22836\2925528738.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  schema_df = pd.read_sql(schema_query, sql_conn)


# 7. Define data type mappings

In [157]:
print("=" * 65)
print("DATA TYPE MAPPING")
print("=" * 65)

DATA TYPE MAPPING


In [158]:
type_mapping = {
    'int': 'INTEGER',
    'bigint': 'BIGINT',
    'smallint': 'SMALLINT',
    'tinyint': 'SMALLINT',
    'bit': 'BOOLEAN',
    'decimal': 'NUMERIC',
    'numeric': 'NUMERIC',
    'money': 'NUMERIC(19,4)',
    'smallmoney': 'NUMERIC(10,4)',
    'float': 'DOUBLE PRECISION',
    'real': 'REAL',
    'datetime': 'TIMESTAMP',
    'datetime2': 'TIMESTAMP',
    'smalldatetime': 'TIMESTAMP',
    'date': 'DATE',
    'time': 'TIME',
    'char': 'CHAR',
    'varchar': 'VARCHAR',
    'nchar': 'CHAR',
    'nvarchar': 'VARCHAR',
    'text': 'TEXT',
    'ntext': 'TEXT'
}

In [159]:
print("SQL Server to PostgreSAL type mapping ")
print()

for sql_type, pg_type in list(type_mapping.items()):
    print(f"    {sql_type:100}  --->     {pg_type}")

SQL Server to PostgreSAL type mapping 

    int                                                                                                   --->     INTEGER
    bigint                                                                                                --->     BIGINT
    smallint                                                                                              --->     SMALLINT
    tinyint                                                                                               --->     SMALLINT
    bit                                                                                                   --->     BOOLEAN
    decimal                                                                                               --->     NUMERIC
    numeric                                                                                               --->     NUMERIC
    money                                                                                         

## 8. Create tables in PostgreSQL  

In [160]:
print("=" * 65)
print("CREATE TABLES IN POSTGRES")
print("=" * 65)

CREATE TABLES IN POSTGRES


In [161]:
try:
    for table in tables_to_migrate:

        schema = table_schema[table]

        pg_table = table.lower()

        pg_cursor.execute(f"DROP TABLE IF EXISTS {pg_table} CASCADE")

        column_definitions = []

        for idx, row in schema.iterrows():
            col_name = row['COLUMN_NAME'].lower()
            sql_type = row['DATA_TYPE']

            base_type = sql_type.lower()
            pg_type =type_mapping.get(base_type, 'TEXT')

            condition_1 = idx == 0                      # Must be first column in the table
            condition_2 = col_name.endswith('id')       # Must end with ID
            condition_3 = 'int' in sql_type.lower()     # Must be INT data type


            if condition_1 and condition_2 and condition_3:
                column_definitions.append(f"{col_name} SERIAL PRIMARY KEY")
            else:
                column_definitions.append(f"{col_name} {pg_type}")
        
        
        column_string = ",\n        ".join(column_definitions)
        create__query = f"""
        CREATE TABLE {pg_table} (
            {column_string}
        )
        """

        pg_cursor.execute(create__query)
        pg_conn.commit()

    print("\n + " + "=" * 55)
    print("[SUCCESS] ---> All tables created successfully!")


except psycopg2.Error as e:
    print(f"Postgres experienced an error while creating a table: {e}")
    pg_conn.rollback()
    raise

except Exception as e:
    print(f"Unexpected issue: {e}")
        


 + =======================================================
[SUCCESS] ---> All tables created successfully!


# 9. Test migration with one table

In [1]:
print("=" * 65)
print("TESTING MIGRATION (SINGLE TABLE)")
print("=" * 65)

TESTING MIGRATION (SINGLE TABLE)


In [3]:
test_table = 'Categories'
pg_table = test_table.lower()


In [ ]:
try:
    print("1. Read from SQL Server... ")
    extract_query = f"SELECT * FROM {pg_table}"
    test_df = pd.read_sql(extract_query, sql_conn)

    print(f"        Read {len(test_df)} rows")


    print("Transforming data types...")

    if 'IsActive' in test_df.columns:
        test_df['IsActive'] = test_dff['IsActive'].astype('bool')
        print("[SUCCESS] ---> Converted IsActive: BIT ---> BOOLEAN")
    

    
    print("3. Prepare the data for loading")
    data_tuples = [tuple(row) for row in test_df.to_numpy()]

    columns = [col.lower() for col in test_df.columns]

    column_string = ', '.join(columns)

    placeholers = ', '.join(['%s'] * len(columns))

    insert_query = f"""
        INSERT INTO {pg_table} ({column_string})
        VALUES %s
    """

    print("4. Insert data into PostgreSQL...")
    execute_values(pg_cursor, insert_query, data_tuples, page_size=1000)
    pg_conn.commit()
"""
    
except Exception as e:
    pass]

